## **Course-Implement Named Entity Recognition with BERT by Hiren Gadhvi**

In [ ]:
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.8 MB/s eta 0:00:00

NER Data: https://www.kaggle.com/datasets/rajnathpatel/ner-data
    
Annotated Corpus for Named Entity Recognition - https://www.kaggle.com/datasets/abhinavwalia95/entity-annotated-corpus

In [ ]:
ls -ltr

total 14856
drwxr-xr-x 1 root root     4096 Dec 19 14:20 sample_data/
-rw-r--r-- 1 root root 15208151 Jan  1 13:02 ner_dataset.csv


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv("ner_dataset.csv",encoding="latin1")

In [ ]:
data.head(30)

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
5,NaN,through,IN,O
6,NaN,London,NNP,B-geo
7,NaN,to,TO,O
8,NaN,protest,VB,O
9,NaN,the,DT,O


In [ ]:
data = data.fillna(method="ffill")

<ipython-input-6-c269da69c753>:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method="ffill")


In [ ]:
data.head(30)

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O
5,Sentence: 1,through,IN,O
6,Sentence: 1,London,NNP,B-geo
7,Sentence: 1,to,TO,O
8,Sentence: 1,protest,VB,O
9,Sentence: 1,the,DT,O


## **Demo: Data Preprocessing**

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


1.   LabelEncoder():
  *   Creates an instance of LabelEncoder from sklearn.preprocessing.
  *   This encoder is used to convert categorical labels (strings or objects) into numeric values.
2.  .fit_transform(data["Sentence #"]):
  *   Fits the LabelEncoder on the Sentence # column of the data DataFrame.
  *   The Sentence # column likely contains categorical data, such as identifiers for sentences (e.g., "Sentence: 1", "Sentence: 2").
  *   Assigns a unique numeric label to each unique category (e.g., "Sentence: 1" -> 0, "Sentence: 2" -> 1).
3. data["Sentence #"] = ...:
  *  Updates the Sentence # column in the DataFrame with the numeric labels.

**Purpose:**
The line converts the Sentence # column, which is probably a categorical identifier, into numeric values. This transformation is necessary for many machine learning algorithms, which typically require numeric inputs rather than categorical strings.

In [ ]:
data["Sentence #"] = LabelEncoder().fit_transform(data["Sentence #"])

In [ ]:
data.head(30)

,Sentence #,Word,POS,Tag
0,0,Thousands,NNS,O
1,0,of,IN,O
2,0,demonstrators,NNS,O
3,0,have,VBP,O
4,0,marched,VBN,O
5,0,through,IN,O
6,0,London,NNP,B-geo
7,0,to,TO,O
8,0,protest,VB,O
9,0,the,DT,O


In [ ]:
# rename the column labels
data.rename(columns={"Sentence #":"sentence_id","Word":"words","Tag":"labels"},inplace=True)

In [ ]:
data["labels"] = data["labels"].str.upper()

In [ ]:
X= data[["sentence_id","words"]]
Y = data["labels"]

In [ ]:
# test size is 20%. can vary to fine-tune the accuracy
x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.2)

## **Demo: BERT Model Training and Evaluation for NER**

Now we train the model and evaluate it.  we'll try and extract relevant entities or tagging for the geopolitical data using BERT

In [ ]:
# building up train data and test data
train_data = pd.DataFrame({"sentence_id":x_train["sentence_id"],"words":x_train["words"],"labels":y_train})
test_data  = pd.DataFrame({"sentence_id":x_test["sentence_id"],"words":x_test["words"],"labels":y_test})

In [ ]:
train_data['labels'].unique()

array(['O', 'B-GPE', 'B-GEO', 'I-GEO', 'I-ART', 'I-TIM', 'B-TIM', 'I-PER',
       'I-ORG', 'B-ORG', 'B-PER', 'B-EVE', 'B-NAT', 'I-GPE', 'I-EVE',
       'B-ART', 'I-NAT'], dtype=object)

These labels are typical in **Named Entity Recognition (NER)** tasks, a common task in Natural Language Processing (NLP). Here's what they represent:

### Format:
- **B-**: Beginning of an entity.
- **I-**: Inside an entity.
- **O**: Outside of any entity (not part of a named entity).

### Common Entity Types:
- **GEO**: Geographical locations (e.g., countries, cities, landmarks).
- **GPE**: Geopolitical entities (e.g., nations, cities, government regions).
- **PER**: Persons (e.g., names of people).
- **ORG**: Organizations (e.g., companies, agencies, institutions).
- **ART**: Artifacts (e.g., titles of books, works of art).
- **EVE**: Events (e.g., sports events, wars, conferences).
- **NAT**: Natural phenomena (e.g., hurricanes, rivers).
- **TIM**: Time expressions (e.g., dates, times, durations).

### Examples:
1. **Sentence**: "Barack Obama was born in Hawaii."
   - `Barack`: **B-PER** (beginning of a person entity).
   - `Obama`: **I-PER** (inside a person entity).
   - `Hawaii`: **B-GEO** (beginning of a geographical location entity).
   - Other words: **O** (outside any entity).

2. **Sentence**: "The United Nations is headquartered in New York."
   - `United`: **B-ORG** (beginning of an organization entity).
   - `Nations`: **I-ORG** (inside the organization entity).
   - `New`: **B-GEO** (beginning of a geographical location entity).
   - `York`: **I-GEO** (inside the geographical location entity).
   - Other words: **O**.

These labels follow the **IOB format** (Inside, Outside, Beginning), which is widely used for tagging entities in sequences.

train_data

In [ ]:
train_data

,sentence_id,words,labels
908959,35023,body,O
327007,5526,.,O
461985,12373,a,O
272057,2727,to,O
479364,13254,the,O
...,...,...,...
1010560,40243,several,O
377778,8095,suburb,O
821040,30586,to,O
29812,3746,Red,I-ORG


In [ ]:
from simpletransformers.ner import NERModel,NERArgs

The line:

```python
from simpletransformers.ner import NERModel, NERArgs
```

is used to import components for building and configuring a Named Entity Recognition (NER) model using the **Simple Transformers** library. Here's what each component does:

### 1. **`NERModel`**
   - This is the main class for creating and working with a NER model.
   - It abstracts much of the complexity of working with Transformer-based models like BERT, RoBERTa, DistilBERT, etc.
   - You can use it to:
     - Initialize a pre-trained NER model.
     - Train a NER model on your dataset.
     - Evaluate the model's performance.
     - Make predictions on new data.

   **Example Usage:**
   ```python
   model = NERModel("bert", "bert-base-cased", args=ner_args)
   ```

   In this example:
   - `"bert"` specifies the model type.
   - `"bert-base-cased"` specifies the pre-trained model to use.

---

### 2. **`NERArgs`**
   - This is a class for defining the configuration and arguments for the NER model.
   - It allows you to specify parameters such as:
     - Learning rate.
     - Number of epochs.
     - Batch size.
     - Output directory for saving the model.
     - Logging options, etc.

   **Example Usage:**
   ```python
   ner_args = NERArgs()
   ner_args.num_train_epochs = 3
   ner_args.train_batch_size = 16
   ner_args.output_dir = "outputs/"
   ```

---

### Typical Workflow:
1. **Import Required Classes:**
   ```python
   from simpletransformers.ner import NERModel, NERArgs
   ```

2. **Set Up Arguments:**
   ```python
   ner_args = NERArgs()
   ner_args.num_train_epochs = 3
   ner_args.train_batch_size = 16
   ner_args.output_dir = "outputs/"
   ```

3. **Initialize the Model:**
   ```python
   model = NERModel("bert", "bert-base-cased", args=ner_args)
   ```

4. **Train the Model:**
   ```python
   model.train_model(train_data)
   ```

5. **Evaluate the Model:**
   ```python
   result, model_outputs, predictions = model.eval_model(eval_data)
   ```

6. **Make Predictions:**
   ```python
   predictions, raw_outputs = model.predict(["This is a sample sentence."])
   ```

---

### Why Use `simpletransformers`?
- It simplifies the process of working with complex Transformer models.
- You can easily perform tasks like NER without needing deep knowledge of libraries like Hugging Face Transformers.
- It's great for quick prototyping or for those who need a high-level API for NLP tasks.

In [ ]:
label = train_data["labels"].unique().tolist()
label

['O',
 'B-GPE',
 'B-GEO',
 'I-GEO',
 'I-ART',
 'I-TIM',
 'B-TIM',
 'I-PER',
 'I-ORG',
 'B-ORG',
 'B-PER',
 'B-EVE',
 'B-NAT',
 'I-GPE',
 'I-EVE',
 'B-ART',
 'I-NAT']

### What is a Hyperparameter?

A **hyperparameter** is a configuration parameter set before training a machine learning model. These parameters are not learned from the data but instead guide the learning process. They influence the behavior, performance, and results of the training process. Examples include learning rate, number of training epochs, and batch size.

---

### Explanation of Hyperparameters in Your Code:

1. **`args.num_train_epochs = 1`**
   - **What it does**: Specifies the number of passes through the entire training dataset during model training.
   - **Impact**:
     - Fewer epochs (e.g., 1) might lead to underfitting (the model doesn't learn enough).
     - More epochs (e.g., 10) might lead to overfitting (the model memorizes the training data but fails to generalize).
   - **Tuning**: Adjust based on the dataset size and complexity.

---

2. **`args.learning_rate = 1e-4`**
   - **What it does**: Determines the step size for updating the model's weights during optimization.
   - **Impact**:
     - A smaller learning rate (e.g., `1e-5`) allows for more fine-tuned updates but may take longer to converge.
     - A larger learning rate (e.g., `1e-2`) speeds up training but may overshoot the optimal point or result in instability.
   - **Tuning**: It's often a good practice to start with a default like `1e-4` or `5e-5` for Transformer models and adjust based on performance.

---

3. **`args.overwrite_output_dir = True`**
   - **What it does**: If `True`, allows overwriting the contents of the output directory during training.
   - **Impact**:
     - Ensures that previous models, logs, or outputs in the specified directory are replaced by the new training process.
     - If `False`, the process will throw an error if the directory isn't empty.
   - **When to Use**: Set to `True` if you're okay with replacing old outputs.

---

4. **`args.train_batch_size = 32`**
   - **What it does**: Defines the number of samples processed at once during training.
   - **Impact**:
     - A larger batch size (e.g., `64`) allows for smoother gradient updates but requires more memory.
     - A smaller batch size (e.g., `16`) may lead to noisier gradient updates but is less memory-intensive.
   - **Tuning**: Choose based on available GPU/CPU memory. A batch size of `16` or `32` is common for Transformer models.

---

5. **`args.eval_batch_size = 32`**
   - **What it does**: Defines the number of samples processed at once during evaluation or prediction.
   - **Impact**:
     - Similar to `train_batch_size`, a larger batch size speeds up evaluation but requires more memory.
     - Smaller batch sizes might be slower but work better on systems with limited resources.
   - **Tuning**: Usually set equal to or smaller than `train_batch_size`.

---

### Summary of Hyperparameters in Context
These hyperparameters together define **how the model trains and evaluates**:
- `num_train_epochs`: Controls the duration of training.
- `learning_rate`: Adjusts the sensitivity of the weight updates.
- `overwrite_output_dir`: Ensures new results are saved without errors.
- `train_batch_size`: Affects training efficiency and memory usage.
- `eval_batch_size`: Affects evaluation efficiency and memory usage.

Proper tuning of these parameters is essential for achieving good model performance without wasting computational resources or running into hardware limitations.

In [ ]:
args = NERArgs()
args.num_train_epochs = 1
args.learning_rate = 1e-4
args.overwrite_output_dir = True
args.train_batch_size = 32
args.eval_batch_size = 32

In [ ]:
model = NERModel("bert", "bert-base-cased",labels=label,args=args)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
model.train_model(train_data,eval_data = test_data,acc=accuracy_score)

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/simpletransformers/ner/ner_model.py:758: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 1:   0%|          | 0/1499 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/simpletransformers/ner/ner_model.py:782: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


(1499, 0.19814091714830936)

In [ ]:
result, modeling_outputs, predictions = model.eval_model(test_data)

  0%|          | 0/3 [00:00<?, ?it/s]

Running Evaluation:   0%|          | 0/1463 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/simpletransformers/ner/ner_model.py:1303: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [ ]:
result

{'eval_loss': 0.17169612143710714,
 'precision': 0.8305944481117327,
 'recall': 0.763265436434455,
 'f1_score': 0.7955078612428251}

These results provide insights into the performance of your Named Entity Recognition (NER) model on the evaluation dataset. Here's what each metric indicates:

---

### 1. **`eval_loss`**
   - **Value**: `0.16949162623168876`
   - **What it means**: This is the average loss on the evaluation dataset, which measures how well the model's predictions align with the true labels.
     - A lower loss indicates better performance, but it must be interpreted along with other metrics like precision, recall, and F1-score.
     - Loss alone doesn’t provide a complete picture of model performance, especially in imbalanced datasets.

---

### 2. **`precision`**
   - **Value**: `0.8316769536562402` (83.17%)
   - **What it means**: Precision is the proportion of correctly predicted entities out of all entities the model predicted.
     - **Formula**: \( \text{Precision} = \frac{\text{True Positives}}{\text{True Positives + False Positives}} \)
     - A high precision means the model makes fewer false positive predictions (e.g., predicts entities that don't exist).

---

### 3. **`recall`**
   - **Value**: `0.7630046708042741` (76.30%)
   - **What it means**: Recall is the proportion of correctly predicted entities out of all actual entities in the dataset.
     - **Formula**: \( \text{Recall} = \frac{\text{True Positives}}{\text{True Positives + False Negatives}} \)
     - A high recall means the model successfully identifies most of the entities but may include some incorrect ones.

---

### 4. **`f1_score`**
   - **Value**: `0.795862184032702` (79.59%)
   - **What it means**: The F1-score is the harmonic mean of precision and recall, providing a balanced measure of both.
     - **Formula**: \( \text{F1-score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision + Recall}} \)
     - It is particularly useful when there is an imbalance between false positives and false negatives.

---

### Interpretation:
- **Precision (83.17%)**: The model is relatively good at predicting correct entities without introducing many incorrect ones.
- **Recall (76.30%)**: The model misses some entities but performs reasonably well at identifying them.
- **F1-score (79.59%)**: This balanced score suggests the model is performing decently, but there is room for improvement.
- **Eval Loss (0.169)**: Indicates that the model's predictions are relatively aligned with the ground truth during evaluation.

---

### Next Steps:
- **Focus on Recall**: If high recall is more critical (e.g., detecting all critical entities), consider adjustments such as lowering the classification threshold.
- **Optimize Further**:
  - Fine-tune hyperparameters like `learning_rate` or `num_train_epochs`.
  - Use more training data if available.
  - Experiment with different pre-trained models (e.g., `roberta-base`, `bert-large`).

This evaluation indicates a well-trained model, but further tuning might be needed based on your specific application requirements.

In [ ]:
predictions, raw_outputs = model.predict(["What is the new name of Bangalore."])

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/simpletransformers/ner/ner_model.py:1643: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [ ]:
predictions

[[{'What': 'O'},
  {'is': 'O'},
  {'the': 'O'},
  {'new': 'O'},
  {'name': 'O'},
  {'of': 'O'},
  {'Bangalore.': 'B-GEO'}]]

In [ ]:
predictions, raw_outputs = model.predict(["What is the new name of Mysore."])

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
predictions

[[{'What': 'O'},
  {'is': 'O'},
  {'the': 'O'},
  {'new': 'O'},
  {'name': 'O'},
  {'of': 'O'},
  {'Mysore.': 'B-GEO'}]]

In [ ]:
raw_outputs

[[{'What': [[9.41,
     -2.92,
     -0.1611,
     -0.9575,
     -1.04,
     0.815,
     0.7783,
     -1.257,
     0.2417,
     0.3062,
     -2.064,
     -1.813,
     -1.779,
     -3.264,
     -2.145,
     -1.09,
     -2.814]]},
  {'is': [[10.73,
     -2.068,
     -0.4062,
     -1.281,
     -1.404,
     0.352,
     0.747,
     -1.226,
     0.1096,
     -0.675,
     -1.788,
     -2.371,
     -1.864,
     -2.43,
     -2.03,
     -1.457,
     -2.28]]},
  {'the': [[9.64,
     -2.078,
     2.312,
     -0.271,
     -1.563,
     0.05893,
     -0.0967,
     -1.644,
     0.04785,
     -0.02322,
     -2.064,
     -2.572,
     -2.033,
     -2.88,
     -2.322,
     -1.324,
     -2.807]]},
  {'new': [[9.625,
     -1.689,
     0.1353,
     -1.191,
     -1.861,
     1.658,
     3.236,
     -1.444,
     -0.691,
     -0.636,
     -1.628,
     -1.983,
     -1.849,
     -2.668,
     -2.568,
     -1.526,
     -2.5]]},
  {'name': [[9.68,
     -2.898,
     -0.394,
     -0.631,
     -1.26,
     1.187,
     0.